<a href="https://colab.research.google.com/github/minarmaing/Handspell-v03/blob/main/kazim_handspell_video_v01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install the official Kaggle CLI command engine
!pip install -q kaggle

# 2. Upload your kaggle.json file from your computer
from google.colab import files
print("Please select your downloaded kaggle.json file:")
files.upload()

# 3. Create the hidden directory structure required by the CLI and move the token
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
# Set permissions so only you can read your API credential key
!chmod 600 ~/.kaggle/kaggle.json

# 4. Pull down the exact Mediapipe Landmarks dataset directly from Kaggle
print("Downloading landmarks-include dataset from Kaggle...")
!kaggle datasets download -d daskoushik/landmarks-include

# 5. Unzip the dataset into a workspace folder named 'include_landmarks'
print("Extracting files...")
!unzip -q landmarks-include.zip -d include_landmarks/

# 6. Clean up the zip file to optimize your local runtime storage space
!rm landmarks-include.zip

print("Dataset successfully downloaded and extracted into the 'include_landmarks' folder!")

Please select your downloaded kaggle.json file:


Saving access_token to access_token
mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/daskoushik/landmarks-include
License(s): unknown
100% 2.69G/2.69G [00:46<00:00, 62.8MB/s]

Extracting files...
Dataset successfully downloaded and extracted into the 'include_landmarks' folder!


In [2]:
import os

# List the contents of the main folder to see if it's organized by CSVs or subfolders
print("Main directory folders:", os.listdir('include_landmarks'))


Main directory folders: ['landmark_files_all']


In [3]:
import pandas as pd
import glob

# Find all Parquet files in the extracted directory
parquet_files = glob.glob('include_landmarks/**/*.parquet', recursive=True)

if parquet_files:
    # Read just the first few rows of one file to inspect the structure
    sample_df = pd.read_parquet(parquet_files[0])
    print(f"Successfully loaded: {parquet_files[0]}")
    print(f"Dataframe Shape: {sample_df.shape}")
    print("\nAvailable columns:")
    print(list(sample_df.columns)[:20]) # Print first 20 columns to avoid overwhelming the screen
else:
    print("No .parquet files found. Double-check your extraction directory path.")


Successfully loaded: include_landmarks/landmark_files_all/Grey/5212.parquet
Dataframe Shape: (32037, 7)

Available columns:
['type', 'landmark_index', 'x', 'y', 'z', 'video_id', 'frame']


In [4]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def load_and_format_sequences(parquet_path, target_sequence_length=30):
    print("Loading Parquet data file into memory...")
    df = pd.read_parquet(parquet_path)

    # 1. Filter out everything except hand data to speed up LSTM training
    # (Adjust 'right_hand' / 'left_hand' based on what unique strings exist in your 'type' column)
    hand_df = df[df['type'].str.contains('hand', case=False, na=False)].copy()

    # Create a unique coordinate label identifier (e.g., "right_hand_0_x")
    hand_df['feature_name'] = hand_df['type'] + "_" + hand_df['landmark_index'].astype(str)

    # 2. Pivot the data: convert long landmark index rows into wide coordinate columns
    print("Pivoting data structure from long to wide format...")
    pivoted = hand_df.pivot_table(
        index=['video_id', 'frame'],
        columns='feature_name',
        values=['x', 'y', 'z']
    )

    # Flatten multi-index columns down to a simple list: ['x_right_hand_0', 'y_right_hand_0', ...]
    pivoted.columns = [f"{col[0]}_{col[1]}" for col in pivoted.columns]
    pivoted = pivoted.reset_index()

    # Fill any missing detection gaps with zero tracking points
    pivoted = pivoted.fillna(0.0)

    feature_cols = [c for c in pivoted.columns if c not in ['video_id', 'frame']]
    num_features = len(feature_cols)
    print(f"Detected {num_features} total coordinate input features per frame.")

    X_sequences = []
    y_labels = []

    # 3. Group frames sequentially by their unique video ID track
    print("Grouping frames into chronological time-series matrices...")
    grouped = pivoted.groupby('video_id')

    for video_id, frame_data in tqdm(grouped):
        # Sort sequentially by chronological frame order
        frame_data = frame_data.sort_values('frame')
        features_matrix = frame_data[feature_cols].to_numpy()

        # Enforce unified sequence length tracking rules
        if len(features_matrix) >= target_sequence_length:
            # Clip sequences that are too long
            features_matrix = features_matrix[:target_sequence_length]
        else:
            # Pad sequences that are too short with zero arrays
            padding_len = target_sequence_length - len(features_matrix)
            padding = np.zeros((padding_len, num_features))
            features_matrix = np.vstack((features_matrix, padding))

        X_sequences.append(features_matrix)

        # For demonstration, dummy label placeholder mapping
        # You will link video_id to your metadata dictionary mapping text classes later
        y_labels.append(0)

    return np.array(X_sequences), np.array(y_labels), num_features

# Run the formatter processing pipeline over your sample file
X_data, y_data, feature_count = load_and_format_sequences('include_landmarks/landmark_files_all/Grey/5212.parquet')
print("\nFinal Input Tensor shape for LSTM processing:", X_data.shape)
# Resulting Shape: (Number_of_Videos, 30_Frames, Feature_Count)


Loading Parquet data file into memory...
Pivoting data structure from long to wide format...
Detected 126 total coordinate input features per frame.
Grouping frames into chronological time-series matrices...


100%|██████████| 1/1 [00:00<00:00, 198.69it/s]


Final Input Tensor shape for LSTM processing: (1, 30, 126)


In [ ]:
print("Your model's expected feature layout order:")
print(feature_cols)

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical

# Setup target parameter tracking variables
SEQUENCE_LENGTH = 30
TOTAL_CLASSES = 24  # Your ASL classification letters count target
y_categorical = to_categorical(y_data, num_classes=TOTAL_CLASSES)

# Instantiate the sequential memory pipeline
model = Sequential([
    # Layer 1: First LSTM layer processing time-series sequences.
    # 'return_sequences=True' is required to pass data forward to the next LSTM layer.
    LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, feature_count)),
    BatchNormalization(),
    Dropout(0.2),

    # Layer 2: Second LSTM layer capturing longer-term sequence patterns
    # 'return_sequences=False' condenses the output down to a 2D matrix
    LSTM(128, return_sequences=False),
    BatchNormalization(),
    Dropout(0.2),

    # Layer 3: Classic dense network processing
    Dense(64, activation='relu'),
    Dropout(0.3),

    # Output Layer: Softmax probability distribution map over target letters
    Dense(TOTAL_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(model.summary())


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        48,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 30, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,296 (618.34 KB)

 Trainable params: 157,912 (616.84 KB)

 Non-trainable params: 384 (1.50 KB)

None


In [8]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm

def load_all_dataset_sequences(dataset_root_dir, target_sequence_length=30):
    # Find EVERY parquet file across all subdirectories
    all_parquet_paths = glob.glob(os.path.join(dataset_root_dir, '**/*.parquet'), recursive=True)
    print(f"Discovered {len(all_parquet_paths)} total Parquet storage shards.")

    # Quick sanity boundary check
    if len(all_parquet_paths) == 0:
        raise FileNotFoundError(f"No .parquet data arrays discovered under path: {dataset_root_dir}")

    # Define standard set of features to ensure consistent shape across all files:
    # 21 landmarks for left_hand and 21 landmarks for right_hand with x, y, z coordinates = 126 features
    standard_features = []
    for hand in ['left_hand', 'right_hand']:
        for i in range(21):
            for coord in ['x', 'y', 'z']:
                standard_features.append(f"{coord}_{hand}_{i}")

    X_global_list = []
    y_global_list = []
    feature_count = len(standard_features)

    print("Beginning structural multi-file data loading matrix construction...")
    # Loop over every file path found (limiting to a safe subset for testing)
    for path in tqdm(all_parquet_paths[:10]):
        try:
            df = pd.read_parquet(path)

            # Filter for hand tracking coordinates
            hand_df = df[df['type'].str.contains('hand', case=False, na=False)].copy()
            if hand_df.empty:
                continue

            hand_df['feature_name'] = hand_df['type'] + "_" + hand_df['landmark_index'].astype(str)

            # Pivot from long to wide structural parameters
            pivoted = hand_df.pivot_table(index=['video_id', 'frame'], columns='feature_name', values=['x', 'y', 'z'])
            pivoted.columns = [f"{col[0]}_{col[1]}" for col in pivoted.columns]
            pivoted = pivoted.reset_index()

            # Reindex columns to match standard features list, filling missing ones with 0.0
            for col in standard_features:
                if col not in pivoted.columns:
                    pivoted[col] = 0.0
            pivoted = pivoted.fillna(0.0)

            # Group timeline by explicit video tracks
            grouped = pivoted.groupby('video_id')
            for video_id, frame_data in grouped:
                frame_data = frame_data.sort_values('frame')
                matrix = frame_data[standard_features].to_numpy()

                # Apply padding / truncating sequence boundary rules
                if len(matrix) >= target_sequence_length:
                    matrix = matrix[:target_sequence_length]
                else:
                    padding = np.zeros((target_sequence_length - len(matrix), feature_count))
                    matrix = np.vstack((matrix, padding))

                X_global_list.append(matrix)

                # Mock label mapping mapping assignment logic:
                mock_label_index = int(video_id) % 24 if str(video_id).isdigit() else 0
                y_global_list.append(mock_label_index)

        except Exception as e:
            # Safely skip corrupt data file layers
            continue

    return np.array(X_global_list), np.array(y_global_list), feature_count

# Trigger the updated global data gathering loop
X_data, y_data, feature_count = load_all_dataset_sequences('include_landmarks/landmark_files_all/')
print(f"\nFinal Consolidated Tensor dimensions layout: {X_data.shape}")

Discovered 4278 total Parquet storage shards.
Beginning structural multi-file data loading matrix construction...


100%|██████████| 10/10 [00:00<00:00, 23.48it/s]


Final Consolidated Tensor dimensions layout: (10, 30, 126)


In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
import os

# Ensure output directory exists
os.makedirs('models', exist_ok=True)

# 1. Convert the current y_data (now with 10 samples) to categorical mapping
y_categorical = to_categorical(y_data, num_classes=TOTAL_CLASSES)
print(f"Prepared target labels with shape: {y_categorical.shape}")

# 2. Setup adaptive training hooks to prevent overfitting
callbacks = [
    # Stops training early if validation loss stops improving for 7 consecutive epochs
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    # Automatically backs up only your highest-performing model snapshot
    ModelCheckpoint('models/handspell_lstm_v1.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

# 3. Run the model compilation fitting cycles
history = model.fit(
    X_data, y_categorical,
    epochs=50,
    batch_size=32,
    validation_split=0.2, # Reserves 2 out of the 10 loaded sequences for validation
    callbacks=callbacks,
    shuffle=True
)

print("\nModel training phase completed successfully!")

Prepared target labels with shape: (10, 24)
Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.0000e+00 - loss: 3.1709
Epoch 1: val_loss improved from None to 3.22179, saving model to models/handspell_lstm_v1.keras

Epoch 1: finished saving model to models/handspell_lstm_v1.keras
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.0000e+00 - loss: 3.1709 - val_accuracy: 0.0000e+00 - val_loss: 3.2218
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.0000e+00 - loss: 3.7136
Epoch 2: val_loss improved from 3.22179 to 3.21798, saving model to models/handspell_lstm_v1.keras

Epoch 2: finished saving model to models/handspell_lstm_v1.keras
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.0000e+00 - loss: 3.7136 - val_accuracy: 0.0000e+00 - val_loss: 3.2180
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.0000e+00 - loss: 2.8815
Epoch 3: val_loss improved from 3.21798 to 3.20855, saving model to models/handspell_lstm_v1.keras

Epoch 3: finished saving mo

In [10]:
import json
import os

# 1. Create a local models directory inside Colab
os.makedirs('models', exist_ok=True)

# 2. Save the trained Keras LSTM model architecture and weights
model.save('models/handspell_lstm_v1.keras')
print("Model saved successfully as a .keras bundle.")

# 3. Create and save the character lookup mapping
# Note: Ensure this list matches your training folder index order exactly!
class_mapping = {
    "classes": [chr(i) for i in range(65, 89)] + ["Hello", "Thank You", "Sorry", "Yes", "No"]
}

with open('models/handspell_lstm_v1_metadata.json', 'w') as f:
    json.dump(class_mapping, f, indent=4)

print("Lookup metadata saved successfully as a .json file.")


Model saved successfully as a .keras bundle.
Lookup metadata saved successfully as a .json file.


In [11]:
from google.colab import auth
auth.authenticate_user()

BUCKET_NAME = "handspell-asl-models-bucket"

# Sync your new files up to your bucket root securely
!gsutil cp models/handspell_lstm_v1.keras gs://{BUCKET_NAME}/
!gsutil cp models/handspell_lstm_v1_metadata.json gs://{BUCKET_NAME}/

print("All time-series model assets backed up to GCP Storage!")

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://models/handspell_lstm_v1.keras [Content-Type=application/octet-stream]...
- [1 files][  1.9 MiB/  1.9 MiB]                                                
Operation completed over 1 objects/1.9 MiB.                                      
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://models/handspell_lstm_v1_metadata.json [Content-Type=application/json]...
-
Operation completed over 1 objects/421.0 B.                                      
All time-series model assets backed up to GCP Storage!
